# Rekommendations sökmotor.

I denna är det meningen att jag skall skapa en rekommendations motor till filmer genom att använda olika metoder så som knn, TF-IDF och Bayesian Average.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import normalize

movies = pd.read_csv(r"C:\\Users\\matti\\Downloads\\movies.csv")
ratings = pd.read_csv (r"C:\\Users\\matti\\Downloads\\ratings.csv")
tags = pd.read_csv (r"C:\\Users\\matti\\Downloads\\tags.csv")


Här gör vi det gamla vanliga och laddar in filerna och de bibliotek som vi skall använda. Vi laddar in dem för att senare kunna bearbeta dem.

In [ ]:
movies ['genres'] = movies['genres'].str.split('|')
expand_movies = movies.explode('genres')
explore_movies = pd.get_dummies(expand_movies['genres'])
matrix_movies = explore_movies.groupby(expand_movies['movieId']).sum()
avg_rate = ratings.groupby('movieId')['rating'].mean().reset_index()
count_rate = ratings.groupby('movieId')['rating'].count().reset_index(name= 'count_rate')
movie_stat = avg_rate.merge(count_rate, on= 'movieId')
model_movie = NearestNeighbors (metric='cosine', algorithm='brute')
model_movie.fit (matrix_movies)
snitt = movie_stat['rating'].mean()
röst = movie_stat['count_rate'].quantile(0.90)

Här börjar vi med att se över filerna arbetar med dem för att göra det begripligt för programmet och senare för att datorn skall kunna arbeta med det vi ger den. Vi har kunnat se att rating.csv innehåller ca 30 milj. rader. Detta kräver allt för mycket av datorn så det valdes att gallra i listan och göra den mer fokuserad. Detta för att anpassa oss till de begränsningar som vi har i RAM. Något som i även kan se är att Vi skapade en hot-one encoding i Genre. Detta skapar en matris där vi tilldelar de olika genrena med värden 1 eller 0 (ja eller nej). På detta sett kan datorn nu börja jämföra filmernas genre med varandra och vi har där en bas för att leta efter liknande filmer senare. Vi letar även efter medel i rating och letar efter nearest neighbors. 

Vi delar senare upp rating så att vi kan arbeta med dem och sammanlänkar alla med filmernas ID för att vi skall ha en gemensam punkt.

In [ ]:
def search_movie(title_search, movies):
    result = movies[movies['title'].str.contains(title_search, case=False, na= False, regex=False)]

    if result.empty:
        print ('Ingen film med denna title hittades')
        return None
    return result


def recommend_genre (movie_id, n=5):
    if movie_id not in matrix_movies.index:
        print(f'{movie_id} genre är okänd')
        return pd.DataFrame()
    
    n= max (1,min(n, len(matrix_movies)-1))
    idx = matrix_movies.index.get_loc(movie_id)

    distances, indices = model_movie.kneighbors(matrix_movies.iloc[idx].values.reshape(1,-1),
                                          n_neighbors = n+1)

    similar_ids = matrix_movies.index[indices.flatten()[1:]]
    similarities = 1- distances.flatten()[1:]

    recom = movies[movies['movieId'].isin(similar_ids)][['movieId','title']]
    recom = recom.drop_duplicates('movieId')
    
    recom = recom.merge(pd.DataFrame({'movieId': similar_ids, 'Similar films': similarities}), on = 'movieId', how = 'left'
                        ).sort_values('Similar films', ascending=False).reset_index(drop=True)
    return recom

Här kommer vi till två sökmotorerna. Vi har redan bearbetat filerna och skall nu arbeta med dem. Vi har överst själva sökningen på inputen. Vi strippar den och tar bort allt som kan göra sökningarna kan visa error.

In [ ]:
clean_tags = tags[['movieId','tag']].dropna().copy()
clean_tags['tag'] = clean_tags['tag'].astype(str).str.strip().str.lower()
tag_agg = clean_tags.groupby('movieId')['tag'].apply(lambda x:' '.join(x.unique())).reset_index()

movies_tags = movies[['movieId', 'title']].merge(tag_agg,on='movieId',how= 'left').fillna({'tag': ''})

tfidf = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(movies_tags['tag'])

model_tags = NearestNeighbors(metric='cosine', algorithm='brute')
model_tags.fit(tfidf_matrix)


def recommend_tags (movie_id, n =5):
    id_to_pos = {mid: i for i, mid in enumerate(movies_tags['movieId'].values)}
    if movie_id not in id_to_pos:
        print(f"{movie_id} saknar en sk. tag")
        return pd.DataFrame()
    
    pos = id_to_pos[movie_id]
    n= max(1, min(n, tfidf_matrix.shape[0]-1))

    distances, indices = model_tags.kneighbors(tfidf_matrix[pos], n_neighbors= n+1)

    neighbor_pos = indices.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]

    neighbor_ids = movies_tags.iloc [neighbor_pos]['movieId'].values
    recs = movies [movies['movieId'].isin(neighbor_ids)][['movieId', 'title']].drop_duplicates('movieId')
    recs = recs.merge(pd.DataFrame({'movieId':neighbor_ids, 'similarity':similarities}), on= 'movieId', how= 'left'
                      ).sort_values('similarity', ascending= False).reset_index (drop=True)
    return recs 

Ovan här så städar vi tags genom att vi tar (som innan) bort tomma fält och skapar en arbetskopia. allt stripas och omvandlas till småbokstäver även här för att vi skall kunna motverka driftsstörningar. Även här grupperas allt med movieId för att den skall förbli navet i systemet. Här tar vi in vår tfidf för att vi skall kunna få ett bättre resultat.

I recommend_tags så skall vår TF-IDF inte bara räkna orden utan även lägga mer vikt på sällsynta ord. Detta är något som vi gör i ett försök att få sökningen så precis som mökjligt. För att sedan låta den leta efter grannar. 

In [ ]:
ids_genre = set (matrix_movies.index)
ids_tags = set (movies_tags['movieId'])
common_ids = sorted(list(ids_genre & ids_tags))

genre_matrix_common = matrix_movies.loc[common_ids].values

pos_map = {mid: i for i, mid in enumerate(movies_tags['movieId'].values)}
rows= [pos_map[mid] for mid in common_ids]
tfidf_common = tfidf_matrix[rows]

genre_norm = normalize(genre_matrix_common, norm = 'l2', axis=1)

w_tags=0.7
w_genre=0.3

genre_sparse = csr_matrix (genre_norm*w_genre)
tfidf_scaled= tfidf_common*w_tags
hybrid_matrix = hstack([tfidf_scaled, genre_sparse])
model_hybrid = NearestNeighbors(metric='cosine', algorithm='brute')
model_hybrid.fit(hybrid_matrix)

def recommend_both(movie_id, n=5):
    try:
        pos = common_ids.index(movie_id)
    except ValueError:
        print (f'{movie_id} har tyvärr inga gemansamma tag <-> genre')
        return pd.DataFrame()
    
    n = max(1, min(n, len (common_ids) -1 ))

    distances, indices = model_hybrid.kneighbors(hybrid_matrix.getrow(pos), n_neighbors= n +1)
    neighbor_pos = indices.flatten()[1:]
    similarities = 1 - distances.flatten()[1:]

    neighbor_ids = np.array(common_ids)[neighbor_pos]
    recs = movies[movies['movieId'].isin(neighbor_ids)][['movieId', 'title']].drop_duplicates('movieId')
    sim_df = pd.DataFrame({'movieId': neighbor_ids, 'similarity': similarities})
    recs = recs.merge(sim_df, on= 'movieId')
    recs = recs.merge(movie_stat[['movieId', 'weighted_rating']], on= 'movieId', how='left')
    recs = recs.sort_values(by=['weighted_rating', 'similarity'], ascending= False).reset_index(drop=True)
    return recs

Vi sätter en parameter här där vi låter Tags stå för 70% av vad vi söker, emot genrens 30% vi gör detta i hopp om att kunna få en så bra uppfattning om vad det är som personen söker efter, då genre kan vara oklar och väldigt bred. så blir det en mer pålitlig sökning.

In [ ]:
def bayesian_avg(row):
    v = row['count_rate']
    R = row['rating']
    return (v / (v + röst) * R) + (röst / (v + röst) * snitt)

movie_stat['weighted_rating'] = movie_stat.apply(bayesian_avg, axis=1)

def from_movie_result(results, preferred_title=None):
    if len(results) == 1:
        return results.iloc[0]['movieId']

    if preferred_title:
        exact= results[results['title'].str.lower()==preferred_title.lower()]
        if len(exact)== 1:
            return exact.iloc[0]['movieId']
    
    print ('\nDet finns fler filmer som matchar din sökning.')
    for i, row in results.iterrows():
        print(f"[{i}] {row['title']} (id: {row['movieId']})")

    while True:
        try:
            val= int (input ("skriv nummret för filmen du vill välja.\n>"))
            if val in results.index:
                return results.loc[val, 'movieId']
            else:
                print (f'index {val} finns inte i listan.')
        except:
            pass
            print('Detta var ett ogiltigt val. Försök igen.')

Vi skapar vår Bayesian average uträkning. Detta görs för att vi vill inte att skall förblinda oss av rating utan även se till omfånget av dess omröstning.Vi talar även om för programmet att antalet röster väger tyngre än vad själva ratingen. pga att om vi har en röst som är 5 så skall den inte väga lika mycket som en film som har 4.8 men 30 röster. pga att den senare är en mer pålitlig slutsats.

In [ ]:
def recommend_from_title (title_search, n=5, mode='genre'):
    results = search_movie(title_search, movies)
    if results is None or len(results)==0:
        print('Vi hittade inga filmer till dessa speciofikationer')
        return pd.DataFrame()
    
    movie_id = from_movie_result (results, preferred_title = title_search)
    if movie_id is None:
        print ('Vi kunde inte välja film.')
        return pd.DataFrame()
    
    print (f"Rekommendationerna vi har tagit fram för dig under dina preferencer är.\n{movies.loc[movies['movieId'] == movie_id, 'title'].values[0]}(ID: {movie_id})")

    if mode == 'genre':
        return recommend_genre (movie_id, n=n)
    elif mode == 'tags':
        return recommend_tags(movie_id,n=n)
    elif mode == 'hybrid':
        return recommend_both(movie_id,n=n)
    else:
        raise ValueError ('Mode måste vara ett val mellan "Genre", "Tags" eller "Hybrid"')

In [ ]:
if __name__ == '__main__':
    title_search = input ('söka efter film\n>')
    choise = input('Vad är det du sökningen basseras på?\n Välj en av förjande.\n [1] Filmens Genre?\n [2] Vad som skrivits om den (tags)?\n [3] Eller dess en blandning av både filmens genre och dess "tags \n> ')

    val_sök = {'1': 'genre', '2': 'tags', '3':'hybrid'}
    val_sök = val_sök.get(choise,'hybrid')

    print (f'Påbörjar sökning, letar efter liknande filmer med {val_sök}')

    recommendations = recommend_from_title(title_search, n=10, mode= val_sök)

    if not recommendations.empty:
        print (f'Dina tio rekommenderade filmerna är ({val_sök}):')
        print(recommendations[['title', 'similarity', 'weighted_rating']])
    else:
        print('Kunde tyvärr inte hitta några rekomendationer.')


I detta projektet valdes det att bygga en sökmotor som skulle vara så mångfaciterad som möjligt. Att ta tillvara på såväl tags som rating så att sökningarna blev så robusta som möjligt. Att använda oss av både tags och ganre så kan vi skapa en sökmotor som är mer exakt och låter användaren hitta ett mer anpassat förslag. En av de konsekvenser som dyker upp med att vi använder en Bayesian är att filmer som har mindre inteaktioner kommer att nervärderas. Detta betyder att filmer som är nya eller obskyra inte kommer att få möjlighet att sýnas lika mycket som större mer välkända filmer.